# CS 691 · Module 11 · Building and checking a plan

Trace a plan from its starting facts to a goal, find the first step that cannot run, and repair the order when possible.

**Before you begin.** You can check a claim against a record for the correct item and time. You can read a rule with two required facts.

A cabinet opens only when it is unlocked. A note proposes “open it, then unlock it.” If it starts locked, where does the plan first fail?

<details><summary>Reveal the answer</summary>

At open. The required state, unlocked, does not hold before that action. Unlocking first could supply what opening requires.

</details>

## A goal describes the result you want

A plan is an ordered sequence of actions intended to reach a goal. The state is the collection of facts currently true in our small model. A goal states what should be true at the end. We will plan for one student, Ava, and one camera, C1, so each short name in the state refers to that pair or item.

Suppose Ava has completed orientation and C1 is available. Her goal is to have C1 issued to her. For this exercise the starting state is fully specified. An omitted fact, such as issued, starts false.

**Predict before running the cell.** If the starting state already included issued, how many actions would be needed for this particular goal?

In [ ]:
state = {"oriented", "available"}
goal = {"issued"}
print("goal reached" if goal.issubset(state) else "goal open")

<details><summary>Reveal output and explanation</summary>

```text
goal open
```

Zero. The empty sequence already ends in a state that satisfies the goal.

</details>

## Describe when an action can run

A precondition is a fact an action requires before it can run. An effect is a change the action makes. In this desk model, reserving C1 requires available. It removes available and adds reserved for Ava. Other state facts carry forward. The difference and union operations on sets implement those two changes.

The camera becomes unavailable to other borrowers as soon as Ava reserves it. Keeping it available after the reservation would allow a second borrower to claim the same item.

**Predict before running the cell.** Which fact from the start survives the reservation, and why?

In [ ]:
state = {"oriented", "available"}
required = {"available"}
if required.issubset(state):
    state = (state - {"available"}) | {"reserved"}
print(sorted(state))

<details><summary>Reveal output and explanation</summary>

```text
['oriented', 'reserved']
```

Oriented survives because reserve neither removes it nor changes whose orientation the model tracks.

</details>

## Stop at the first unmet condition

The issue action requires both reserved and oriented. It removes reserved and adds issued. To check a proposed plan, read the actions left to right. Inspect a step’s preconditions in the current state before applying its effects. The first failed step tells you what went wrong.

A sample language-model plan proposes issue followed by reserve. Both are actions the desk knows. Their order prevents execution from this starting state.

**Predict before running the cell.** Does seeing reserve later in the plan satisfy the missing condition at step one?

In [ ]:
actions = {"reserve": ({"available"}, {"available"}, {"reserved"}),
           "issue": ({"reserved", "oriented"}, {"reserved"}, {"issued"})}
state = {"available", "oriented"}
for action in ["issue", "reserve"]:
    needs, removes, adds = actions[action]
    missing = needs - state
    if missing:
        print(action, "needs", ", ".join(sorted(missing)))
        break
    state = (state - removes) | adds

<details><summary>Reveal output and explanation</summary>

```text
issue needs reserved
```

No. Step one sees only the starting facts. A later action cannot establish a fact that an earlier action already needed.

</details>

## Choose an order that reaches the goal

Planning searches for actions that could reach a goal. Verification checks a particular proposed sequence. With just two distinct actions, try both possible orders and verify each. A sequence succeeds only when every action runs and the final state includes the goal.

Reserving first supplies the missing fact for issue. If Ava has yet to complete orientation, swapping these two actions cannot repair the missing orientation.

**Predict before running the cell.** Would the one-step sequence reserve pass a check that tests only whether every step ran?

In [ ]:
from itertools import permutations
actions = {"reserve": ({"available"}, {"available"}, {"reserved"}),
           "issue": ({"reserved", "oriented"}, {"reserved"}, {"issued"})}
for order in permutations(actions):
    state = {"available", "oriented"}
    for name in order:
        needs, removes, adds = actions[name]
        if not needs.issubset(state):
            break
        state = (state - removes) | adds
    else:
        if "issued" in state:
            print(" then ".join(order))

<details><summary>Reveal output and explanation</summary>

```text
reserve then issue
```

Yes, reserve can run. A separate final check rejects it for this goal because the ending state contains reserved and still lacks issued.

</details>

## Follow one reservation through a state change

The state is a set of facts about this case. The subset check asks whether every required fact is present before the action runs. The minus sign removes available, the vertical bar adds reserved, and oriented carries over.

Replace this line:
state = {"oriented", "available"}
with:
state = {"oriented"}
What will print?

In [ ]:
state = {"oriented", "available"}
required = {"available"}
if required.issubset(state):
    state = (state - {"available"}) | {"reserved"}
print(sorted(state))

<details><summary>Worked answer after your change</summary>

['oriented']

The required available fact is missing, so the update is skipped and the original state prints.

</details>

## Exercise 1

A projector is available. Its checkout requires a reservation and an ID check. Reserve requires available. Check ID has no preconditions and adds id_checked. Give two valid orders ending with checkout.

<details><summary>Hint</summary>

Look for two actions that do not depend on one another.

</details>

<details><summary>Worked answer</summary>

Check ID → reserve → checkout and reserve → check ID → checkout both work. The final action needs both earlier effects. Neither earlier action requires the other’s result.

</details>

## Exercise 2

After reserve, a student’s model lists both available and reserved. Which action description would you inspect, and how could the mistake affect a second borrower?

<details><summary>Hint</summary>

Track the removed facts as well as the added ones.

</details>

<details><summary>Worked answer</summary>

Inspect the effects of reserve. For this single-unit camera, reserve should remove available and add reserved. Leaving available in the state would let a second borrower pass the availability test.

</details>

## Exercise 3

A proposed plan reaches issued in the simulation. When Ava arrives, a clerk finds that C1 has been sent for repair. What did the verification establish and what should happen now?

<details><summary>Hint</summary>

Separate the assumed starting facts from observations at the desk.

</details>

<details><summary>Worked answer</summary>

Verification established that the plan works under its starting facts and action definitions. Refresh the state to include the repair and remove availability as appropriate. Then check whether a feasible plan exists, perhaps for a different item if Ava agrees.

</details>

You can verify a proposed action sequence. How should the desk explain a decision to Ava when some requirements have evidence and others are still unknown?

A simulated issue action requires a reservation and orientation. Ava’s reservation is present, but her orientation record has yet to be retrieved. Which check would make the desk’s answer more informative?

<details><summary>Reveal the answer</summary>

Look up Ava’s orientation. The reservation already satisfies one requirement, and the missing record leaves the other unresolved. There is no basis yet for saying she failed orientation.

</details>